In [ ]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWFjaGEvRHJvcGJveC8wNF90ZWFjaGluZy9BUjEwNDk4X0VNYUMxL1Byb2dQeVF1YXJ0by9Ob3RlYm9va3M='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

In [ ]:
if 10 > 5: print("It's true!")

In [ ]:
# YOUR CODE HERE


In [ ]:
if 10 > 5:
    print("Guess What?")
    print("It's true!")
    
print("This is outside the conditional code block")

In [ ]:
if 10 > 15:
    print("Guess What?")
    print("It's true!")
    
print("This is outside the conditional code block")

In [ ]:
print(10 < 5)
print(2 < 5 < 10)
print(10 == 10)
print(10 != 10)

In [ ]:
# YOUR CODE HERE


In [ ]:
print( (2 < 5) & (10 < 20) )

In [ ]:
print(2<10, 50>100)


print( (2 < 10) & (50 > 100) )

In [ ]:
print( (2 < 10) | (50 > 100) )

In [ ]:
a = 2

if a > 5:
    print("a is greater than 5")
else:
    print("a is 5 or less")

In [ ]:
#a=2
# YOUR CODE HERE


if a > 5:
    print("a is greater than 5")
else:
    print("a is 5 or less")

In [ ]:
x = 4

if x > 0:
    print("Positive")
else:
    if x < 0:
        print("Negative")
    else:
        print("Zero")

In [ ]:
import math

#x = 4
# YOUR CODE HERE



if x > 0:
    print("Positive")
else:
    if x < 0:
        print("Negative")
    else:
        print("Zero")

In [ ]:
x = -4

if x > 0:
    print("Positive")
elif x < 0:
    print("Negative")
else:
    print("Zero")

In [ ]:
x = 3

if x < -4:
    y = -4.0
#elif ???:
# YOUR CODE HERE

    y = (x + 4)**2/4.0 - 4.0
#elif ???:
#    y = ???
# YOUR CODE HERE

#else:
#    ???
# YOUR CODE HERE



print(x, y)

In [ ]:
x=0

if x > 0:
    sign = "Positive!"
if x < 0:
    sign = "Negative!"

print(sign)

In [ ]:
x=0

if x > 0:
    sign = "Positive!"
if x < 0:
    sign = "Negative!"
# YOUR CODE HERE


print(sign)

In [ ]:
# Fibonacci sequence
a0 = 1
a1 = 1
a2 = a1+a0
a3 = a2+a1
a4 = a3+a2
a5 = a4+a3

print(a0, a1, a2, a3, a4, a5)

In [ ]:
a = 1
a = a+1
print(a)

a = a/4.0
print(a)

In [ ]:
number = 1

while number <= 10: # check the condition to see whether to skip or run the block
    square = number**2    # the block is run if the condition is true
    print(number, square) # each line is executed in order
    number = number + 1   # the values being compared MUST CHANGE or the block will loop forever! 
    # at the end of the block the condition is checked again and the program moves on if False

print('end of loop')

In [ ]:
# set first value
a0 = 1
print(a0)

# set second value
a1 = 1
print(a1)

# calculate third value
a2 = a1+a0
print(a2)

# put the following block in a while loop to iterate then stop when a goes over 100
# YOUR CODE HERE


In [ ]:
x=1 # initial value for x
print(x**2)

i=0 # set a counter
while i<10:
    x = 0.5*(x + 2/x)
    print(x**2) # show how close x is to sqrt(2)
    i=i+1

In [ ]:
x=1 # initial value for x
print(x**2)

i=0 # set a counter
while i<10:
    x_old = x
    x = 0.5*(x_old + 2/x_old)
    difference = x - x_old
    print(x**2, difference) # print the "error" from the true value
    i=i+1

In [ ]:
x=1 # initial value for x
print(x**2)

difference = 10 # set this initially above the tolerance

while abs(difference) > 1e-6:
    x_old = x
    x = 0.5*(x_old + 2/x_old)
    difference = x - x_old
    print(x**2, difference) # print the "error" from the true value

In [ ]:
# series pi, using pi/4 = 1 - 1/3 + 1/5 - 1/7 + 1/9 ...
s = 1
print(4*s)

n = -1.0 # set the sign of the next term to be negative
d = 3.0
s = s + n/d  # take old value then add next term in sequence (-1/3.0)
print(4*s)

In [ ]:
n = -1*n  # change the sign of the next term by multiplying by -1
d = d + 2  # add 2 to the denominator
s = s + n/d

print(4*s)

In [ ]:
# series pi, using pi/4 = 1 - 1/3 + 1/5 - 1/7 + 1/9 ...

# set initial values
# YOUR CODE HERE


#calculate first two terms
#s = ...
# YOUR CODE HERE



# set any initial value  for err that is larger than the condition in the while:
#err = ...
# YOUR CODE HERE


# define a counter to count the steps
i = 0  # set the counter i


# start a while loop with a condition based on the absolute error
#while ???:
# YOUR CODE HERE

    # iterate using the pi example above as a model
    # iterating the counter and keeping track of the error (it MUST CHANGE each iteration)
    # YOUR CODE HERE
    

print(4*s, err, i) # print pi, error and number of steps

In [ ]:
def correct_abs(x):
    if x > 0:
        y = x
    elif x < 0:
        y = -x
    elif x == 0:
        y = 0
    return y

print(correct_abs(5))
print(correct_abs(-5))
print(correct_abs(0))

In [ ]:
#def ???:
# YOUR CODE HERE

    #disc = ???
    # YOUR CODE HERE
    
    if disc > 0:
        #x1 = ???
        #x2 = ???
        # YOUR CODE HERE
        
        print("2 roots =", x1, x2)
    #elif ???:
        #???
        #print(???)
    # YOUR CODE HERE
    
    #???
        #???
    # YOUR CODE HERE
    

        
quads(1, 0, -1)
quads(1, 2, 1)
quads(1, 0, 1)

In [ ]:
# conditionals
a = -1
if (a > 0) & (type(a) == int):
    print("positive integer")
elif (a < 0) | (type(a) != int):
    print("negative or not an integer")
else:
    print("zero!")
    
#while loops
def lessthanten(x):
    n=0
    while x>=10:
        n+=1
        x=x/10
        print(x, end=' ')
    print() #blank line
    return str(n)+" iterations"

start=1000000
y=lessthanten(start)
print(y)